### Задание

Создайте систему компьютерного зрения, которая будет определять тип геометрической фигуры. Используя подготовленную базу и шаблон ноутбука проведите серию экспериментов по перебору гиперпараметров нейронной сети, распознающей три категории изображений (треугольник, круг, квадрат).

1. Поменяйте количество нейронов в сети, используя следующие значения:

- один слой 10 нейронов
- один слой 100 нейронов
- один слой 5000 нейронов.

2. Поменяйте активационную функцию в скрытых слоях с `relu` на `linear`.
3. Поменяйте размеры batch_size:
- 10
- 100
- 1000

4. Выведите на экран получившиеся точности.

Всего должно получиться 18 комбинаций указанных параметров.

Создайте сравнительную таблицу по результатам проведенных тестов.

In [3]:
# Подключение класса для создания нейронной сети прямого распространения
from tensorflow.keras.models import Sequential
# Подключение класса для создания полносвязного слоя
from tensorflow.keras.layers import Dense
# Подключение оптимизатора
from tensorflow.keras.optimizers import Adam
# Подключение утилит для to_categorical
from tensorflow.keras import utils
# Подключение библиотеки для загрузки изображений
from tensorflow.keras.preprocessing import image
# Подключение библиотеки для работы с массивами
import numpy as np
# Подключение библиотек для отрисовки изображений
import matplotlib.pyplot as plt
# Подключение модуля для работы с файлами
import os
# Вывод изображения в ноутбуке, а не в консоли или файле
%matplotlib inline

## 1. Подготовка данных и путей

В папке с ноутбуком лежит архив **`hw_light.zip`**.  
Далее архив распаковывается в папку `hw_light`, а переменная `base_dir` указывает на директорию с классами.

Особенность структуры: после распаковки фактическая папка с данными находится по пути:

- `base_dir = hw_light/hw_light`

Классы представлены подпапками:

- `0` → класс **0**
- `3` → класс **1**
- `4` → класс **2**

In [13]:
# Распаковываем архив hw_light.zip в папку hw_light
!unzip -q hw_light.zip

In [21]:
# Путь к директории с базой
base_dir = 'hw_light/hw_light'
# Создание пустого списка для загрузки изображений обучающей выборки
x_train = []
# Создание списка для меток классов
y_train = []
# Задание высоты и ширины загружаемых изображений
img_height = 20
img_width = 20
# Перебор папок в директории базы
for patch in os.listdir(base_dir):
    # Перебор файлов в папках
    for img in os.listdir(base_dir + '/' + patch):
        # Добавление в список изображений текущей картинки
        x_train.append(image.img_to_array(image.load_img(base_dir + '/' + patch + '/' + img,
                                                    target_size=(img_height, img_width),
                                                    color_mode='grayscale')))
        # Добавление в массив меток, соответствующих классам
        if patch == '0':
            y_train.append(0)
        elif patch == '3':
            y_train.append(1)
        else:
            y_train.append(2)

# Преобразование в numpy-массив загруженных изображений и меток классов
x_train = np.array(x_train)
y_train = np.array(y_train)
# Вывод размерностей
print('Размер массива x_train', x_train.shape)
print('Размер массива y_train', y_train.shape)

Размер массива x_train (302, 20, 20, 1)
Размер массива y_train (302,)


## 2. Загрузка изображений в массивы

Каждое изображение:
- загружается в оттенках серого (**grayscale**),
- приводится к размеру **20×20**,
- преобразуется в массив NumPy размера `(20, 20, 1)`.

Формируем:
- `x` — массив изображений,
- `y` — метки классов (0/1/2), полученные из названия папки (`0`, `3`, `4`).

In [23]:
import os
import numpy as np
from tensorflow.keras.preprocessing import image

x = []
y = []

img_height = 20
img_width = 20

for patch in os.listdir(base_dir):
    patch_path = os.path.join(base_dir, patch)
    if not os.path.isdir(patch_path):
        continue

    for img_name in os.listdir(patch_path):
        img_path = os.path.join(patch_path, img_name)

        x.append(
            image.img_to_array(
                image.load_img(
                    img_path,
                    target_size=(img_height, img_width),
                    color_mode="grayscale"
                )
            )
        )

        # метки классов (0->0, 3->1, 4->2)
        if patch == "0":
            y.append(0)
        elif patch == "3":
            y.append(1)
        else:  # "4"
            y.append(2)

x = np.array(x, dtype="float32")
y = np.array(y, dtype="int32")

print("x:", x.shape)  # (N,20,20,1)
print("y:", y.shape)  # (N,)
print("labels:", sorted(set(y)))

x: (302, 20, 20, 1)
y: (302,)
labels: [0, 1, 2]


### Проверка размеров

Ожидаемые формы:
- `x`: `(N, 20, 20, 1)`
- `y`: `(N,)`

Где `N` — количество изображений в датасете.

## 3. Предобработка и подготовка выборок

Выполняем:
1. **Нормализацию** пикселей: `x = x / 255.0`  
2. **Flatten**: перевод `(20,20,1)` → `(400,)` для подачи в Dense слой  
3. Перевод меток в **one-hot** (3 класса)  
4. Разделение на выборки:
   - train: 80%
   - test: 20%

In [35]:
from sklearn.model_selection import train_test_split
from tensorflow.keras import utils

x = x / 255.0
x = x.reshape(x.shape[0], -1)     # (N, 400)

y_cat = utils.to_categorical(y, 3)

x_train, x_test, y_train, y_test = train_test_split(
    x, y_cat, test_size=0.2, random_state=42, stratify=y
)

print("x_train:", x_train.shape, "x_test:", x_test.shape)

x_train: (241, 400) x_test: (61, 400)


## 4. Построение модели (Dense)

Архитектура модели:

- `Input(400)` — вход: изображение 20×20, развёрнутое в вектор из 400 значений  
- `Dense(n_units, activation=...)` — скрытый слой (число нейронов и активация меняются в экспериментах)
- `Dense(3, activation="softmax")` — выходной слой на 3 класса

Компиляция:
- optimizer: **Adam**
- loss: **categorical_crossentropy**
- metric: **accuracy**

In [37]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input
from tensorflow.keras.optimizers import Adam

def build_model(n_units, activation_hidden):
    model = Sequential([
        Input(shape=(x_train.shape[1],)),       # 400
        Dense(n_units, activation=activation_hidden),
        Dense(3, activation="softmax")
    ])
    model.compile(optimizer=Adam(),
                  loss="categorical_crossentropy",
                  metrics=["accuracy"])
    return model

## 5. Эксперименты (18 запусков)

Перебираем гиперпараметры:

- `neurons` ∈ {10, 100, 5000}
- `activation` ∈ {relu, linear}
- `batch_size` ∈ {10, 100, 1000}

Итого: **3 × 2 × 3 = 18** экспериментов.

Для каждого эксперимента сохраняем:
- `test_accuracy` и `test_loss`
- `train_acc_last` и `val_acc_last` (последняя эпоха)

In [39]:
import pandas as pd

neurons_list = [10, 100, 5000]
activations = ["relu", "linear"]
batch_sizes = [10, 100, 1000]

EPOCHS = 15

results = []

for n_units in neurons_list:
    for act in activations:
        for bs in batch_sizes:
            model = build_model(n_units, act)

            history = model.fit(
                x_train, y_train,
                epochs=EPOCHS,
                batch_size=bs,
                verbose=0,
                validation_split=0.2
            )

            test_loss, test_acc = model.evaluate(x_test, y_test, verbose=0)

            results.append({
                "neurons": n_units,
                "activation": act,
                "batch_size": bs,
                "test_accuracy": float(test_acc),
                "test_loss": float(test_loss),
                "val_acc_last": float(history.history["val_accuracy"][-1]),
                "train_acc_last": float(history.history["accuracy"][-1]),
            })

            print(f"done: neurons={n_units:>4} act={act:>6} bs={bs:>4} -> test_acc={test_acc:.4f}")

df = pd.DataFrame(results).sort_values("test_accuracy", ascending=False)
df

done: neurons=  10 act=  relu bs=  10 -> test_acc=0.6393
done: neurons=  10 act=  relu bs= 100 -> test_acc=0.5410
done: neurons=  10 act=  relu bs=1000 -> test_acc=0.4590
done: neurons=  10 act=linear bs=  10 -> test_acc=0.5738
done: neurons=  10 act=linear bs= 100 -> test_acc=0.5902
done: neurons=  10 act=linear bs=1000 -> test_acc=0.5738
done: neurons= 100 act=  relu bs=  10 -> test_acc=0.7049
done: neurons= 100 act=  relu bs= 100 -> test_acc=0.7213
done: neurons= 100 act=  relu bs=1000 -> test_acc=0.6721
done: neurons= 100 act=linear bs=  10 -> test_acc=0.6230
done: neurons= 100 act=linear bs= 100 -> test_acc=0.5738
done: neurons= 100 act=linear bs=1000 -> test_acc=0.4590
done: neurons=5000 act=  relu bs=  10 -> test_acc=0.7541
done: neurons=5000 act=  relu bs= 100 -> test_acc=0.7213
done: neurons=5000 act=  relu bs=1000 -> test_acc=0.6230
done: neurons=5000 act=linear bs=  10 -> test_acc=0.5246
done: neurons=5000 act=linear bs= 100 -> test_acc=0.5738
done: neurons=5000 act=linear b

,neurons,activation,batch_size,test_accuracy,test_loss,val_acc_last,train_acc_last
12,5000,relu,10,0.754098,0.646134,0.816327,0.989583
7,100,relu,100,0.721311,0.760506,0.755102,0.838542
13,5000,relu,100,0.721311,0.720970,0.693878,0.859375
6,100,relu,10,0.704918,0.720866,0.714286,0.942708
8,100,relu,1000,0.672131,0.754194,0.693878,0.807292
0,10,relu,10,0.639344,0.795756,0.734694,0.864583
14,5000,relu,1000,0.622951,0.933304,0.714286,0.765625
9,100,linear,10,0.622951,1.196588,0.653061,0.901042
4,10,linear,100,0.590164,0.871079,0.653061,0.723958
5,10,linear,1000,0.573770,0.876490,0.571429,0.750000


## 6. Результаты экспериментов


In [42]:
pivot = df.pivot_table(
    index=["neurons", "activation"],
    columns="batch_size",
    values="test_accuracy"
)
pivot

batch_size              10        100       1000
neurons activation                              
10      linear      0.573770  0.590164  0.573770
        relu        0.639344  0.540984  0.459016
100     linear      0.622951  0.573770  0.459016
        relu        0.704918  0.721311  0.672131
5000    linear      0.524590  0.573770  0.557377
        relu        0.754098  0.721311  0.622951

In [52]:
df.to_csv("results_lite.csv", index=False)
pivot.to_csv("results_lite_pivot.csv")
print("Сохранено: results_lite.csv и results_lite_pivot.csv")

Сохранено: results_lite.csv и results_lite_pivot.csv


## Вывод:

In [47]:
best = df.sort_values("test_accuracy", ascending=False).iloc[0]
print(
    f"Лучшая конфигурация:\n"
    f"- neurons: {best['neurons']}\n"
    f"- activation: {best['activation']}\n"
    f"- batch_size: {best['batch_size']}\n"
    f"- test_accuracy: {best['test_accuracy']:.4f}\n"
    f"- test_loss: {best['test_loss']:.4f}"
)

Лучшая конфигурация:
- neurons: 5000
- activation: relu
- batch_size: 10
- test_accuracy: 0.7541
- test_loss: 0.6461
